In [16]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [17]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [18]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
response is spanish
"""


    menssages = []
    add_user_message(menssages, prompt)
    add_assistant_message(menssages, "```json")
    text = chat(menssages, stop_sequences=["```"])
    return json.loads(text)

In [19]:
dataset = generate_dataset()

#dataset

with open("dataset.json", "w") as f:
    json.dump(dataset, f , indent=2)

In [20]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [21]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [22]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [23]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [24]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Expresi\u00f3n Regular para Validar ARN de AWS\n\nAqu\u00ed te presento varias soluciones, desde la b\u00e1sica hasta la m\u00e1s robusta:\n\n## 1. **Expresi\u00f3n Regular B\u00e1sica**\n\n```regex\n^arn:aws:[a-z0-9\\-]+:[a-z0-9\\-]*:[0-9]{12}:.+$\n```\n\n### Explicaci\u00f3n:\n- `^arn:aws:` - Comienza con \"arn:aws:\"\n- `[a-z0-9\\-]+` - Servicio (letras, n\u00fameros, guiones)\n- `:` - Separador\n- `[a-z0-9\\-]*` - Regi\u00f3n (puede estar vac\u00eda)\n- `:` - Separador\n- `[0-9]{12}` - ID de cuenta (12 d\u00edgitos)\n- `:.+` - Recurso (al menos un car\u00e1cter)\n- `$` - Fin de la cadena\n\n---\n\n## 2. **Expresi\u00f3n Regular Mejorada** \u2b50\n\n```regex\n^arn:aws:[-a-z0-9]*:[-a-z0-9]*:[0-9]{12}:[-a-zA-Z0-9/:._]*$\n```\n\n**Mejoras:**\n- Permite may\u00fasculas en el recurso\n- Permite caracteres especiales comunes: `/`, `.`, `_`\n\n---\n\n## 3. **Expresi\u00f3n Regular Avanzada (M\u00e1s Estricta)**\n\n```regex\n^arn:aws:([a-z0-9]+-)*[a-z0-9]+:[a-z0-9\\-]